In [1]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="Qwen/Qwen3-8B-MLX-4bit",
    base_url="http://10.195.19.15:8000/v1",
    api_key="dummy",
    temperature=0,
    max_tokens=8192,
)

In [2]:
response = model.invoke("/no_think Say 'stack is alive' if you can hear me.")
print(response.content)



Stack is alive.


### Config now from `.env`, not hardcoded

`graph.py`/`documents.py` read model/endpoint config via `python-dotenv` — see `.env.example` for the template.

In [2]:
import re

from finanalyticsagent.graph import model as env_model

masked = re.sub(r"(\d+)\.\d+\.\d+\.(\d+)", r"\1.***.***.\2", env_model.openai_api_base)
print("base_url (masked):", masked)
response = env_model.invoke("/no_think Say 'stack is alive' if you can hear me.")
print(response.content)

base_url (masked): http://10.***.***.15:8000/v1


Stack is alive.


## Step 1: load one CSV and inspect it manually

No agent yet — just looking at the raw data so we know its shape before
deciding what the LLM should see (schema + preview only, never the full table).

In [3]:
import pandas as pd

df = pd.read_csv("bazaar_books/caravan_accounts.csv")

print(df.shape)
df.dtypes

(176, 9)


Realm                   str
Guild_Name              str
Year                  int64
Quarter               int64
Operating_Income    float64
EBITDA              float64
Tax                 float64
Net_Income          float64
GOGS                float64
dtype: object

In [23]:
df.head()

,Realm,Guild_Name,Year,Quarter,Operating_Income,EBITDA,Tax,Net_Income,GOGS
0,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,1,21904.87,26090.31,5855.70,16049.17,75668.54
1,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,2,17681.07,23523.18,3967.18,13713.89,62357.84
2,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,3,20009.92,31315.08,4861.25,15148.67,85116.94
3,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,4,14784.45,22956.75,3998.11,10786.34,41932.53
4,Oasis of Whispering Sands,Djinn-Forged Ironworks,718,1,13575.22,16403.08,3558.74,10016.48,35353.81


## Step 2: build the schema-and-preview system prompt

The LLM never sees the full DataFrame. Instead it sees, once, at the start:
- the schema (column names + dtypes) as a markdown table
- a small preview (first N rows) as markdown-KV (key: value pairs) — chosen
  over a markdown table because research shows markdown-KV gives higher
  comprehension accuracy for small models, at the cost of more tokens
  (acceptable here since it's only a handful of rows).

In [24]:
def build_schema_table(df: pd.DataFrame) -> str:
    """Render column names + dtypes as a markdown table.

    Args:
        df: the DataFrame to describe.

    Returns:
        A markdown table string, one row per column: "column | dtype".
    """
    lines = ["| column | dtype |", "|---|---|"]
    for column_name, dtype in df.dtypes.items():
        lines.append(f"| {column_name} | {dtype} |")
    return "\n".join(lines)


def build_preview_kv(df: pd.DataFrame, n_rows: int = 5) -> str:
    """Render the first n_rows of a DataFrame as markdown-KV blocks.

    Each row becomes a block of "column: value" lines separated by "---".
    Chosen over a markdown table for better small-model comprehension.

    Args:
        df: the DataFrame to preview.
        n_rows: how many rows from the top to include.

    Returns:
        A markdown-KV formatted string.
    """
    blocks = []
    for _, row in df.head(n_rows).iterrows():
        lines = [f"{column_name}: {value}" for column_name, value in row.items()]
        blocks.append("\n".join(lines))
    return "\n---\n".join(blocks)


print(build_schema_table(df))
print()
print(build_preview_kv(df))

| column | dtype |
|---|---|
| Realm | str |
| Guild_Name | str |
| Year | int64 |
| Quarter | int64 |
| Operating_Income | float64 |
| EBITDA | float64 |
| Tax | float64 |
| Net_Income | float64 |
| GOGS | float64 |

Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 1
Operating_Income: 21904.87
EBITDA: 26090.31
Tax: 5855.7
Net_Income: 16049.17
GOGS: 75668.54
---
Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 2
Operating_Income: 17681.07
EBITDA: 23523.18
Tax: 3967.18
Net_Income: 13713.89
GOGS: 62357.84
---
Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 3
Operating_Income: 20009.92
EBITDA: 31315.08
Tax: 4861.25
Net_Income: 15148.67
GOGS: 85116.94
---
Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 4
Operating_Income: 14784.45
EBITDA: 22956.75
Tax: 3998.11
Net_Income: 10786.34
GOGS: 41932.53
---
Realm: Oasis of Whispering Sands
Guild_

In [25]:
SYSTEM_PROMPT_TEMPLATE = """\
You are a financial analytics assistant. You answer questions about a single
pandas DataFrame called `df`, which is already loaded in your execution
environment — you never need to load or recreate it.

You do not have the full table in front of you. You have only the schema
and a small preview below. To answer any question that needs real numbers,
you must call the `execute_python_code` tool with pandas code that operates
on `df` and returns the result. Never guess numeric values — always compute
them via the tool.

If the user asks for a chart, plot, or visualization, use the `create_chart`
tool instead — it runs matplotlib code against `df` and returns the path to
a saved PNG. Do not try to describe a chart in text; use the tool.

## Schema

{schema_table}

## Preview (first {n_preview_rows} rows)

{preview_kv}

## How to answer

- For small talk ("hello", "thank you") — respond directly, do not call the tool.
- For any question needing numbers from the data — write pandas code against
  `df` and call `execute_python_code`. Do not answer from memory or from the
  preview above; the preview is only a sample, not the full data.
- For any question asking for a chart, plot, or visualization — call
  `create_chart` instead.
"""


def build_system_prompt(df: pd.DataFrame, n_preview_rows: int = 5) -> str:
    """Build the full system prompt for the analytics agent.

    Args:
        df: the DataFrame the agent will answer questions about.
        n_preview_rows: how many rows to include in the preview section.

    Returns:
        The rendered system prompt string.
    """
    return SYSTEM_PROMPT_TEMPLATE.format(
        schema_table=build_schema_table(df),
        n_preview_rows=n_preview_rows,
        preview_kv=build_preview_kv(df, n_preview_rows),
    )


system_prompt = build_system_prompt(df)
print(system_prompt)

You are a financial analytics assistant. You answer questions about a single
pandas DataFrame called `df`, which is already loaded in your execution
environment — you never need to load or recreate it.

You do not have the full table in front of you. You have only the schema
and a small preview below. To answer any question that needs real numbers,
you must call the `execute_python_code` tool with pandas code that operates
on `df` and returns the result. Never guess numeric values — always compute
them via the tool.

If the user asks for a chart, plot, or visualization, use the `create_chart`
tool instead — it runs matplotlib code against `df` and returns the path to
a saved PNG. Do not try to describe a chart in text; use the tool.

## Schema

| column | dtype |
|---|---|
| Realm | str |
| Guild_Name | str |
| Year | int64 |
| Quarter | int64 |
| Operating_Income | float64 |
| EBITDA | float64 |
| Tax | float64 |
| Net_Income | float64 |
| GOGS | float64 |

## Preview (first 5 rows)



## Step 3: the `execute_python_code` tool

Same idea as the old OpenAI Code Interpreter workflow this project replaces:
the LLM writes pandas code that ends in `print(...)`, the tool runs that
code against the real `df` with `exec()`, captures whatever was printed to
stdout, and returns it as text. If the code raises an exception, we don't
crash — we return the error message back to the agent so it can see what
went wrong and try again with corrected code.

In [ ]:
import contextlib
import io

from langchain_core.tools import tool

MAX_TOOL_OUTPUT_CHARS = 4000


@tool
def execute_python_code(code: str) -> str:
    """Run pandas code against the loaded financial DataFrame `df` and return its printed output.

    The DataFrame `df` and the `pd` (pandas) module are already available —
    do not try to import pandas or load/recreate `df` yourself.

    Your code MUST call print(...) on whatever value answers the question.
    Anything not printed is lost — this tool only returns what was printed.

    Print only what you need to answer the question (a single value, a small
    aggregate, a short table) — do not print the entire DataFrame. Output is
    truncated past a length limit, since `df` may be much larger in real use.

    Args:
        code: a snippet of Python/pandas code, e.g.
            "print(df.groupby('Realm')['Net_Income'].sum().idxmax())"

    Returns:
        Everything the code printed to stdout, as a single string (truncated
        if too long, with a note telling you to narrow your query). If the
        code raised an exception instead, returns an "Error: ..." message
        describing what went wrong, so you can fix the code and try again.
    """
    namespace = {"df": df, "pd": pd}
    stdout_buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_buffer):
            exec(code, namespace)
    except Exception as e:
        return f"Error: {e}"

    output = stdout_buffer.getvalue()
    if not output:
        return "Code ran without errors but printed nothing. Use print() to show a result."

    if len(output) > MAX_TOOL_OUTPUT_CHARS:
        truncated = output[:MAX_TOOL_OUTPUT_CHARS]  # explain detaile
        return (
            f"{truncated}\n"
            f"...\n"
            f"[Output truncated at {MAX_TOOL_OUTPUT_CHARS} characters — your code printed too "
            f"much. Narrow your query: filter rows first, aggregate instead of printing raw "
            f"rows, or use .head()/.describe() instead of printing the whole result.]"
        )
    return output


In [27]:

# quick manual check — no LLM involved, just calling the tool function directly
print(execute_python_code.invoke({"code": "print(df.groupby('Realm')['Net_Income'].sum().idxmax())"}))

Garden of the Midnight Rose



### Guard test: simulate a huge print (no LLM involved)

Our real dataset is only 176 rows, too small to actually trigger the
truncation naturally. This manually forces a large output (repeating text)
to confirm the guard kicks in correctly before we ever hit it with a real
large table.

In [28]:
huge_print_code = "for i in range(2000): print(f'row {i}: some financial data goes here')"
result = execute_python_code.invoke({"code": huge_print_code})

print(f"Returned length: {len(result)} chars (limit: {MAX_TOOL_OUTPUT_CHARS})")
print("Ends with:")
print(result[-300:])

Returned length: 4215 chars (limit: 4000)
Ends with:
 some financial data goes here
row 104: some financial data goes here
row 105: some f
...
[Output truncated at 4000 characters — your code printed too much. Narrow your query: filter rows first, aggregate instead of printing raw rows, or use .head()/.describe() instead of printing the whole result.]


## Step 4: assemble the agent

`create_agent` wires together the model, the tool, and our system prompt
into a ReAct loop (agent node ↔ tools node, repeating until the model
stops calling tools). This cell only builds the agent object — no LLM
call happens yet. The first real call is the next step.

In [29]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[execute_python_code],
    system_prompt=system_prompt,
)

## Step 5: first end-to-end question

This is the first real call to the LLM through the agent. We ask a
question adapted from the sample queries in the README — instead of
"country" / 2023, we use our synthetic setting's "realm" / year 718.
We print every message in the result so we can see the full ReAct loop:
the AI message with a tool call, the ToolMessage with the tool's output,
and the final AI answer.

In [12]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Which company made the most profit?"}]}
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Which company made the most profit?
================================== Ai Message ==================================
Tool Calls:
  execute_python_code (5f619a5c-e5c7-4ec5-a625-051d855ffe34)
 Call ID: 5f619a5c-e5c7-4ec5-a625-051d855ffe34
  Args:
    code: print(df.groupby('Guild_Name')['Net_Income'].sum().idxmax())
================================= Tool Message =================================
Name: execute_python_code

Vizier's Ledger & Ore Co.

================================== Ai Message ==================================



The company that made the most profit is **Vizier's Ledger & Ore Co.**, based on the highest total Net Income across all quarters.


### Debug: inspect the last AI message raw

`pretty_print()` only shows `.content`. If content is empty, the real
information is elsewhere — either `tool_calls` (model decided to call a
tool but message rendering hid it) or `response_metadata` (e.g. a
`finish_reason: "length"` meaning the model hit `max_tokens` before
producing visible output, likely spent on hidden `<think>...</think>`
reasoning tokens Qwen3 emits by default).

In [11]:
last_ai_message = [m for m in result["messages"] if m.type == "ai"][-1]

print("content:", repr(last_ai_message.content))
print("tool_calls:", last_ai_message.tool_calls)
print("response_metadata:", last_ai_message.response_metadata)

content: "\n\nThe company that made the most profit is **Vizier's Ledger & Ore Co.** This was determined by summing the `Net_Income` for each guild and identifying the one with the highest total."
tool_calls: []
response_metadata: {'token_usage': {'completion_tokens': 365, 'prompt_tokens': 1149, 'total_tokens': 1514, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 1091}}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-8B-MLX-4bit', 'system_fingerprint': '0.31.2-0.31.1-macOS-26.3.1-arm64-arm-64bit-applegpu_g13s', 'id': 'chatcmpl-90689088-7d03-4ea6-82f4-8eb5e6cc81a9', 'finish_reason': 'stop', 'logprobs': None}


## Step 6: test against reference questions (no dedicated tools yet)

Questions adapted from the legacy Forvis Mazars assistant's sample queries
and this project's README, translated to our schema (Realm/Guild_Name,
years 717-718) and skipping anything that needs a chart (no `create_chart`
tool yet). Also includes two small-talk checks to confirm the agent does
NOT call the tool when it doesn't need to.

We're deliberately using only the one generic `execute_python_code` tool —
per the updated roadmap, dedicated tools per operation come later, only if
this generic approach proves insufficient somewhere.

In [36]:
REFERENCE_QUESTIONS = [
    "Which realm had the highest net income in 718?",
    "Which realm had the highest taxes in each quarter of 718?",
    "Which company had the highest operating income growth between quarters of 718?",
    "How has operating income evolved between 717 and 718, and what could be the reasons for this change?",
    "Hello!",
    "Thank you!",
]


def run_question(agent, question: str) -> None:
    """Run one question through the agent and print a compact transparency report.

    Args:
        agent: the compiled agent to invoke.
        question: the natural-language question to ask.
    """
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    print(f"Q: {question}")

    tool_calls_made = [
        (call["name"], call["args"]["code"])
        for message in result["messages"]
        if message.type == "ai"
        for call in message.tool_calls
    ]
    if tool_calls_made:
        for tool_name, code in tool_calls_made:
            print(f"Used {tool_name} with:")
            print(f"  {code}")
    else:
        print("Answered directly, no tool call.")

    final_answer = result["messages"][-1].content
    print(f"A: {final_answer}")
    print("-" * 80)


for question in REFERENCE_QUESTIONS:
    run_question(agent, question)

Q: Which realm had the highest net income in 718?
Used execute_python_code with:
  print(df[df['Year'] == 718].groupby('Realm')['Net_Income'].sum().idxmax())
A: 

The realm with the highest net income in year 718 was **Garden of the Midnight Rose**.
--------------------------------------------------------------------------------
Q: Which realm had the highest taxes in each quarter of 718?
Used execute_python_code with:
  filtered = df[df['Year'] == 718]
grouped = filtered.groupby(['Quarter', 'Realm'])['Tax'].sum().reset_index()
result = grouped.groupby('Quarter').apply(lambda x: x.loc[x['Tax'].idxmax(), 'Realm']).reset_index(drop=True)
print(result)
A: 

For the year 718, the realm with the highest taxes in each quarter was:

- **Quarter 1**: Straits of the Bottled Storm  
- **Quarter 2**: Grove of the Singing Palms  
- **Quarter 3**: Garden of the Midnight Rose  
- **Quarter 4**: Oasis of Whispering Sands  

This result identifies the top taxing realm for each quarter based on total t

### Debug: raw AI messages for the two questions that returned empty

Q2 and Q3 both answered directly (no tool call) with empty visible content.
`finish_reason` tells us why: `"stop"` means the model genuinely finished
with nothing to show (unlikely but possible), `"length"` means it hit
`max_tokens` before producing visible text — almost certainly spent on
hidden `<think>...</think>` reasoning. We just raised `max_tokens` above;
this cell re-runs just these two questions and inspects the raw messages.

**Important:** re-running the `model = ChatOpenAI(...)` cell creates a new
model object, but `agent` was built from the *old* one. Re-run the
`create_agent(...)` cell too before running this, otherwise you're still
testing with the old `max_tokens=2048`.

In [13]:
DEBUG_QUESTIONS = [
    "Which realm had the highest taxes in each quarter of 718?",
    "Which company had the highest operating income growth between quarters of 718?",
]

for question in DEBUG_QUESTIONS:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    last_ai_message = [m for m in result["messages"] if m.type == "ai"][-1]

    print(f"Q: {question}")
    print("content:", repr(last_ai_message.content))
    print("tool_calls:", last_ai_message.tool_calls)
    print("finish_reason:", last_ai_message.response_metadata.get("finish_reason"))
    print("completion_tokens:", last_ai_message.response_metadata.get("token_usage", {}).get("completion_tokens"))
    print("-" * 80)

Q: Which realm had the highest taxes in each quarter of 718?
content: '\n\nFor the year 718, the realm with the highest taxes in each quarter was:\n\n- **Quarter 1**: Straights of the Bottled Storm  \n- **Quarter 2**: Grove of the Singing Palms  \n- **Quarter 3**: Garden of the Midnight Rose  \n- **Quarter 4**: Oasis of Whispering Sands  \n\nThis result identifies the realm with the maximum tax value in each quarter of 718.'
tool_calls: []
finish_reason: stop
completion_tokens: 821
--------------------------------------------------------------------------------
Q: Which company had the highest operating income growth between quarters of 718?
content: '\n\nThe company with the highest operating income growth between quarters of Year 718 is **Forty Thieves Foundry**. \n\nThis was determined by calculating the difference in operating income between consecutive quarters for each guild in Year 718 and identifying the guild with the maximum growth.'
tool_calls: []
finish_reason: stop
complet

## Step 7: test loading an arbitrary user file

So far `df` was hardcoded to one specific CSV. This step checks the whole
pipeline generalizes to a *different* file with a *different* schema — a
stand-in for what a Streamlit file-upload widget will hand us later.

`bazaar_books/guild_ledger.csv` is a second synthetic dataset (same
fictional guilds, different metrics: market share, headcount, customer
satisfaction — no financial columns at all) to prove nothing here is
secretly tied to `Realm`/`Net_Income`/etc.

`load_table` picks CSV vs XLSX by file extension — the minimal version of
the future `read_new_table(path)` tool from the roadmap.

In [ ]:
def load_table(path: str) -> pd.DataFrame:
    """Load a user-provided table file into a DataFrame, by extension.

    Args:
        path: path to a .csv or .xlsx file.

    Returns:
        The loaded DataFrame.

    Raises:
        ValueError: if the file extension is neither .csv nor .xlsx.
    """
    if path.endswith(".csv"):
        return pd.read_csv(path)
    if path.endswith(".xlsx"):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type for {path!r}, expected .csv or .xlsx")


# Reassign the global `df` — execute_python_code looks up `df` fresh from
# the global namespace on every call, so it automatically picks up whichever
# table is currently loaded without needing to be redefined.
df = load_table("bazaar_books/guild_ledger.csv")
print(df.shape)
print(df.dtypes)

# system_prompt and agent were built from the OLD df — rebuild both so the
# schema/preview and the agent's tool loop reflect the new table.
system_prompt = build_system_prompt(df)
agent = create_agent(
    model=model,
    tools=[execute_python_code],
    system_prompt=system_prompt,
)

In [ ]:
run_question(agent, "Which guild has the highest average customer satisfaction score?")

## Step 8: real user-driven file upload (not hardcoded, not a typed path)

Typing a path (previous attempt) is still not a real upload — it just moves
the "who decides the file" problem from code to a text prompt. A real
upload means: the user clicks a button, picks a file through their own
file system's dialog, and the file's bytes get handed to us — exactly how
Streamlit's `st.file_uploader` works.

`ipywidgets.FileUpload` is the Jupyter equivalent: it renders a clickable
button, opens your OS's native file picker, and gives us the raw bytes of
whatever you selected. Two cells, because the upload is asynchronous —
you need to click and pick a file in the first cell's widget before running
the second cell that reads what you picked.

In [20]:
import ipywidgets as widgets
from IPython.display import display

upload_widget = widgets.FileUpload(accept=".csv,.xlsx", multiple=False)
display(upload_widget)

# Click the button above, pick a .csv or .xlsx from your own filesystem,
# then run the next cell.

FileUpload(value=(), accept='.csv,.xlsx', description='Upload')

In [21]:
if not upload_widget.value:
    raise ValueError("No file selected yet — click the button above and choose a file first.")

uploaded_file = upload_widget.value[0]  # tuple of dicts in ipywidgets 8.x
filename = uploaded_file["name"]
file_bytes = io.BytesIO(uploaded_file["content"])

if filename.endswith(".csv"):
    df = pd.read_csv(file_bytes)
elif filename.endswith(".xlsx"):
    df = pd.read_excel(file_bytes)
else:
    raise ValueError(f"Unsupported file type: {filename}")

print(f"Loaded {filename} -> shape {df.shape}")
print(df.dtypes)

system_prompt = build_system_prompt(df)
agent = create_agent(
    model=model,
    tools=[execute_python_code],
    system_prompt=system_prompt,
)

print(
    "\n--- Data hygiene reminder ---\n"
    "Whatever file you just loaded is now baked into this cell's output.\n"
    "This notebook is version-controlled and gets pushed to a public repo:\n"
    "before committing, make sure the file above was a synthetic dataset\n"
    "(bazaar_books/*.csv), not real company data. If it wasn't, reload a\n"
    "synthetic file and re-run this cell, or clear this cell's output first."
)

Loaded caravan_accounts.csv -> shape (176, 9)
Realm                   str
Guild_Name              str
Year                  int64
Quarter               int64
Operating_Income    float64
EBITDA              float64
Tax                 float64
Net_Income          float64
GOGS                float64
dtype: object

--- Data hygiene reminder ---
Whatever file you just loaded is now baked into this cell's output.
This notebook is version-controlled and gets pushed to a public repo:
before committing, make sure the file above was a synthetic dataset
(bazaar_books/*.csv), not real company data. If it wasn't, reload a
synthetic file and re-run this cell, or clear this cell's output first.


### Test the agent on whatever you just uploaded

Generic financial question, phrased without assuming any specific column
names — the agent reads whatever schema `build_system_prompt` generated
for the file you picked in Step 8, and figures out the mapping itself
(same as it did for `Realm`/`Guild_Name` earlier in this notebook).

**Reminder:** if you uploaded a real file, this cell's output will contain
a real result. Do not commit it — before committing, re-run Step 8 with a
synthetic file (or clear this cell's output) so the notebook stays safe
to push.

In [22]:
run_question(agent, "What's the single most important insight you can find in this data?")

Q: What's the single most important insight you can find in this data?
Used execute_python_code with:
  print(df['Net_Income'].sum())
A: 

The total Net Income across all entries is **1,747,657.56**. This represents the cumulative profit after all expenses, taxes, and other deductions, highlighting the overall financial performance of the recorded guilds across the specified time period.
--------------------------------------------------------------------------------


## Step 9: `create_chart` tool

Same shape as `execute_python_code`, but instead of expecting `print(...)`,
the LLM's code is expected to draw a plot using `plt` (matplotlib.pyplot) —
e.g. `plt.bar(...)`, `plt.plot(...)`. The tool then saves whatever is on
the current matplotlib figure to `outputs/` as a PNG and returns the file
path, instead of returning text.

If the code ran but never actually drew anything (no axes on the current
figure), we return an error asking the model to actually call a plotting
function — same "fail loud, let the model retry" pattern as
`execute_python_code`'s error handling.

In [37]:
import uuid
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # no display backend needed, we only save to disk
import matplotlib.pyplot as plt

OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(exist_ok=True)


@tool
def create_chart(code: str) -> str:
    """Run matplotlib plotting code against the loaded DataFrame `df` and save it as a PNG.

    The DataFrame `df`, `pd` (pandas), and `plt` (matplotlib.pyplot) are
    already available — do not import them yourself. Your code must
    actually draw something (e.g. plt.bar(...), plt.plot(...), plt.pie(...))
    using data computed from `df`. Do not call plt.show() or plt.savefig()
    yourself — the tool handles saving.

    Args:
        code: a snippet of Python/pandas/matplotlib code that draws a chart,
            e.g. "df.groupby('Realm')['Net_Income'].sum().plot(kind='bar')"

    Returns:
        The file path of the saved PNG (under outputs/), as a string. If the
        code raised an exception, or ran without drawing anything, returns
        an "Error: ..." message describing what went wrong.
    """
    plt.close("all")  # start from a clean figure each time
    namespace = {"df": df, "pd": pd, "plt": plt}
    try:
        exec(code, namespace)
    except Exception as e:
        return f"Error: {e}"

    fig = plt.gcf()
    if not fig.get_axes():
        return "Error: no chart was drawn. Call a plotting function like plt.bar(...) or plt.plot(...)."

    filename = OUTPUTS_DIR / f"chart_{uuid.uuid4().hex[:8]}.png"
    fig.savefig(filename, bbox_inches="tight")
    plt.close(fig)
    return str(filename)


# quick manual check — no LLM involved, just calling the tool function directly
result = create_chart.invoke(
    {"code": "df.groupby('Realm')['Net_Income'].sum().sort_values().plot(kind='barh', figsize=(8, 6))"}
)
print(result)

outputs/chart_aabed8fd.png


### Wire `create_chart` into the agent and test end-to-end

Rebuild `system_prompt` (now mentions both tools) and `agent` (now has both
tools) against whatever `df` is currently loaded, then ask for a chart.

In [38]:
system_prompt = build_system_prompt(df)
agent = create_agent(
    model=model,
    tools=[execute_python_code, create_chart],
    system_prompt=system_prompt,
)

run_question(agent, "Plot net income by realm as a bar chart.")

Q: Plot net income by realm as a bar chart.
Used create_chart with:
  df.groupby('Realm')['Net_Income'].sum().plot(kind='bar')
A: 

The bar chart showing total Net Income by Realm has been generated and saved as [chart_0751cb29.png](outputs/chart_0751cb29.png). The chart displays the summed Net Income values across all quarters for each unique Realm in the dataset. Would you like me to analyze any specific realm's data or create additional visualizations?
--------------------------------------------------------------------------------


## Step 10: same thing, but via the extracted `.py` modules

Steps 2-9 above defined everything inline in this notebook. After
confirming those steps work correctly, the same logic was extracted into
`finanalyticsagent/` as reusable modules: `active_table.py` (holds the
current DataFrame), `tools.py` (`execute_python_code`, `create_chart`,
`load_table`), `prompts.py` (schema/preview/system prompt), `graph.py`
(`model` + `build_agent(df)`), `testing.py` (`run_question`, the
transparency-report helper originally defined in Step 6).

This cell proves the modules work standalone — it does not depend on any
earlier cell in this notebook having been run, only on the package itself.

In [6]:
from finanalyticsagent.tools import load_table
from finanalyticsagent.graph import build_agent
from finanalyticsagent.testing import run_question

module_df = load_table("bazaar_books/caravan_accounts.csv")
module_agent = build_agent(module_df)

print("Module-based agent built. Testing a text question and a chart question:\n")
run_question(module_agent, "Which realm had the highest net income?")
run_question(module_agent, "Plot EBITDA by realm as a bar chart.")

Module-based agent built. Testing a text question and a chart question:

Q: Which realm had the highest net income?
Used execute_python_code with:
  print(df.groupby('Realm')['Net_Income'].sum().idxmax())
A: 

The realm with the highest net income is **Garden of the Midnight Rose**.
--------------------------------------------------------------------------------
Q: Plot EBITDA by realm as a bar chart.
Used create_chart with:
  df.groupby('Realm')['EBITDA'].sum().plot(kind='bar')
A: 

The bar chart showing EBITDA by realm has been created and saved. You can view it at the following path:

```
outputs/chart_60039062.png
```
--------------------------------------------------------------------------------


## Step 11: automated test suite (pytest)

Every step above was checked by looking at printed output — running a cell,
reading the result, deciding whether it looked right. That does not scale
past a handful of things to remember, and it already missed real bugs at
least twice in this project: the tool-usage report once said
"execute_python_code" even when `create_chart` had actually been called
(caught only by noticing the mismatch by eye), and after an unrelated
prompt edit made to support charts, the small-talk answer started leaking
`DataFrame`/`pandas`/`df` straight into an end-user-facing reply.

A `pytest` suite under `tests/` (kept as its own reusable set of files, not
inline in this notebook — the notebook stays the R&D record, not the test
runner) now guards against this class of regression, in two layers run
separately below.

In [7]:
!python -m pytest tests/test_prompts.py tests/test_tools.py tests/test_active_table.py tests/test_graph.py -v --no-header

============================= test session starts ==============================
collected 8 items                                                              

tests/test_prompts.py::test_build_schema_table_renders_columns_and_dtypes PASSED [ 12%]
tests/test_prompts.py::test_build_preview_kv_renders_rows_as_kv_blocks PASSED [ 25%]
tests/test_tools.py::test_load_table_raises_on_unsupported_extension PASSED [ 37%]
tests/test_tools.py::test_execute_python_code_truncates_past_the_limit PASSED [ 50%]
tests/test_tools.py::test_create_chart_errors_when_nothing_is_drawn PASSED [ 62%]
tests/test_active_table.py::test_get_df_raises_when_nothing_was_set PASSED [ 75%]
tests/test_graph.py::test_detects_a_truncated_answer PASSED              [ 87%]
tests/test_graph.py::test_does_not_flag_a_normal_answer PASSED           [100%]

============================== 8 passed in 1.69s ===============================


### Layer 2: LLM regression checks (requires a reachable mlx_lm.server)

Slower and non-deterministic — a real model answers each question — but
each check is tied to a specific bug that already happened once during
Streamlit review: implementation details leaking into small talk, a raw
file path or "download this file" phrasing leaking into a chart answer, a
missing bold on the key answer value, and a chart PNG that either doesn't
exist or isn't transparent.

In [6]:
!python -m pytest tests/test_agent_regressions.py -v --no-header

============================= test session starts ==============================
collected 4 items                                                              

tests/test_agent_regressions.py::test_small_talk_never_mentions_implementation_details PASSED [ 25%]
tests/test_agent_regressions.py::test_chart_answer_never_leaks_the_raw_file_path PASSED [ 50%]
tests/test_agent_regressions.py::test_final_answer_bolds_the_key_value PASSED [ 75%]
tests/test_agent_regressions.py::test_chart_question_produces_a_real_transparent_png PASSED [100%]

========================= 4 passed in 72.36s (0:01:12) =========================


## Step 12: prototype multi-table access (`dfs['name']`)

So far the agent has only ever seen one DataFrame at a time (`df`). This
step tests whether it can work with several **named** tables at once,
matching how the legacy Assistants API's Code Interpreter actually worked
(every file attached to it sits together in one sandbox) — see CLAUDE.md's
Multi-file support roadmap item for the full phased plan. Nothing above
this cell is touched; this is a new, self-contained prototype, not yet
extracted into `finanalyticsagent/`.

Three named tables are loaded: the two existing synthetic datasets, plus a
third, `realm_metadata`, mapping each `Realm` to a fictional `Region` —
purely synthetic, same convention as the rest of `bazaar_books/`. Crucially,
`realm_metadata` shares no column at all with `guild_ledger` — a question
needing both must go through `caravan_accounts` as a bridge, a transitive
join.

Validated live against the real model before writing this cell (not
guessed): the agent correctly picks the right single table for
single-table questions, and correctly recognizes and executes the
three-table bridge join for a question that genuinely needs it. One real
issue surfaced and is addressed directly in the prompt/tool docstring
below: the model's first attempt at the bridge join merged on only one
shared column (`Guild_Name`) instead of all three (`Guild_Name`, `Year`,
`Quarter`), silently inflating 96 correct rows to 768 — harmless for that
specific `mean()`-based question, but a real risk for `sum()`/`count()`-based
ones.

In [1]:
import contextlib
import io

import pandas as pd
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from finanalyticsagent.prompts import build_schema_table, build_preview_kv

model = ChatOpenAI(
    model="Qwen/Qwen3-8B-MLX-4bit",
    base_url="http://10.195.19.15:8000/v1",
    api_key="dummy",
    temperature=0,
    max_tokens=8192,
)

In [2]:
dfs = {
    "caravan_accounts": pd.read_csv("bazaar_books/caravan_accounts.csv"),
    "guild_ledger": pd.read_csv("bazaar_books/guild_ledger.csv"),
    "realm_metadata": pd.read_csv("bazaar_books/realm_metadata.csv"),
}
for name, table in dfs.items():
    print(name, table.shape)

caravan_accounts (176, 9)
guild_ledger (96, 6)
realm_metadata (22, 2)


In [3]:
MULTI_TABLE_SYSTEM_PROMPT_TEMPLATE = """\
You are a financial analytics assistant. You answer questions using one or
more pandas DataFrames, available in a dict called `dfs` — access a table
as `dfs['table_name']`. You never need to load or recreate them.

Some questions only need one table. Others need you to combine (e.g.
`pd.merge`) two or more tables — check the schemas below for columns the
tables share, and merge on ALL of the columns they have in common (not
just one), to avoid accidentally duplicating rows when multiple rows share
a single column's value. Tables that don't share a column directly may
still be connected through a third table that shares a column with both.

To answer any question that needs real numbers, call `execute_python_code`
with pandas code that operates on `dfs` and returns the result via print().
Never guess numeric values — always compute them via the tool.

{tables_section}

## How to answer

- For small talk — respond directly, do not call the tool.
- For any question needing numbers — write pandas code against `dfs` and
  call `execute_python_code`.
"""


def build_multi_table_prompt(dfs: dict) -> str:
    """Build a system prompt describing several named tables at once.

    Args:
        dfs: mapping of table name to DataFrame.

    Returns:
        The rendered multi-table system prompt string.
    """
    sections = []
    for name, table in dfs.items():
        sections.append(
            f"## Table `{name}` (access as dfs['{name}'])\n\n"
            f"{build_schema_table(table)}\n\n"
            f"Preview (first 5 rows):\n\n{build_preview_kv(table)}"
        )
    return MULTI_TABLE_SYSTEM_PROMPT_TEMPLATE.format(tables_section="\n\n".join(sections))


multi_system_prompt = build_multi_table_prompt(dfs)
print(multi_system_prompt)

You are a financial analytics assistant. You answer questions using one or
more pandas DataFrames, available in a dict called `dfs` — access a table
as `dfs['table_name']`. You never need to load or recreate them.

Some questions only need one table. Others need you to combine (e.g.
`pd.merge`) two or more tables — check the schemas below for columns the
tables share, and merge on ALL of the columns they have in common (not
just one), to avoid accidentally duplicating rows when multiple rows share
a single column's value. Tables that don't share a column directly may
still be connected through a third table that shares a column with both.

To answer any question that needs real numbers, call `execute_python_code`
with pandas code that operates on `dfs` and returns the result via print().
Never guess numeric values — always compute them via the tool.

## Table `caravan_accounts` (access as dfs['caravan_accounts'])

| column | dtype |
|---|---|
| Realm | str |
| Guild_Name | str |
| Year

In [4]:
@tool
def execute_python_code_multi(code: str) -> str:
    """Run pandas code against several loaded DataFrames and return printed output.

    A dict `dfs` mapping table names to DataFrames (and `pd`, the pandas
    module) are already available — access a table as dfs['table_name'].
    Combine tables (e.g. dfs['a'].merge(dfs['b'], on=[...])) if the question
    needs data from more than one — tables not directly sharing a column
    may still be joined through a third table that bridges them. When
    joining two tables, merge on ALL columns they share, not just one, to
    avoid duplicating rows.

    Your code MUST call print(...) on whatever value answers the question.

    Args:
        code: a snippet of Python/pandas code, e.g.
            "print(dfs['caravan_accounts'].groupby('Realm')['Net_Income'].sum().idxmax())"

    Returns:
        Everything the code printed to stdout. If the code raised an
        exception, returns an "Error: ..." message so you can retry.
    """
    namespace = {"dfs": dfs, "pd": pd}
    stdout_buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_buffer):
            exec(code, namespace)
    except Exception as e:
        return f"Error: {e}"

    output = stdout_buffer.getvalue()
    if not output:
        return "Code ran without errors but printed nothing. Use print() to show a result."
    return output

In [5]:
multi_agent = create_agent(
    model=model,
    tools=[execute_python_code_multi],
    system_prompt=multi_system_prompt,
)

### Test: single-table selection, a two-table join, and the transitive bridge join

The first two questions each only need one of the three tables — checking
the agent correctly picks the relevant one and ignores the other two. The
third needs a direct two-table join (`caravan_accounts` + `realm_metadata`
on `Realm`). The fourth genuinely needs all three: `guild_ledger` has no
`Realm` column at all, so answering it requires bridging through
`caravan_accounts` (which has both `Guild_Name` and `Realm`) to reach
`realm_metadata`.

In [6]:
for question in [
    "Which realm had the highest net income?",
    "Which guild has the highest average customer satisfaction score?",
    "Which region has the highest average net income?",
    "Is there a relationship between a guild's region and its customer satisfaction score?",
]:
    result = multi_agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Q: {question}")
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:
                print(f"  code: {call['args'].get('code')}")
    print(f"A: {result['messages'][-1].content.strip()}")
    print("-" * 80)

Q: Which realm had the highest net income?
  code: print(dfs['caravan_accounts'].groupby('Realm')['Net_Income'].sum().idxmax())
A: The realm with the highest net income is **Garden of the Midnight Rose**.
--------------------------------------------------------------------------------
Q: Which guild has the highest average customer satisfaction score?
  code: print(dfs['guild_ledger'].groupby('Guild_Name')['Customer_Satisfaction_Score'].mean().idxmax())
A: The guild with the highest average customer satisfaction score is **Djinn-Forged Ironworks**. This was determined by calculating the mean customer satisfaction score for each guild across all quarters and years, then identifying the guild with the highest average.
--------------------------------------------------------------------------------
Q: Which region has the highest average net income?
  code: merged = pd.merge(dfs['caravan_accounts'], dfs['realm_metadata'], on='Realm')
region_avg = merged.groupby('Region')['Net_Income'].mea

## Step 13: RAG prototype — one tool, naive search, no embeddings

Adds `search_documents` to the existing agent, alongside
`execute_python_code`/`create_chart` — no router or separate agent yet
(an LLM-based router would mean a guaranteed extra model call per turn,
costly for our small local model).

Loaders skip `langchain_community` (unmaintained): `pymupdf` for PDF,
`python-docx` for `.docx`.

Demo files: `zau_al_makan_decree.docx` (a tariff exemption for
Djinn-Forged Ironworks — the same guild name already in the CSVs, to check
the agent picks the right tool despite the overlap) and
`hammam_keeper_proclamation.pdf`.

In [16]:
import pandas as pd
import pymupdf
from docx import Document as DocxDocument
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

from finanalyticsagent import active_table
from finanalyticsagent.prompts import build_system_prompt
from finanalyticsagent.tools import MAX_TOOL_OUTPUT_CHARS, create_chart, execute_python_code

model = ChatOpenAI(
    model="Qwen/Qwen3-8B-MLX-4bit",
    base_url="http://10.195.19.15:8000/v1",
    api_key="dummy",
    temperature=0,
    max_tokens=8192,
)

# Tabular side: unchanged, reuses the already-extracted multi-table modules.
dfs = {
    "caravan_accounts": pd.read_csv("bazaar_books/caravan_accounts.csv"),
    "guild_ledger": pd.read_csv("bazaar_books/guild_ledger.csv"),
    "realm_metadata": pd.read_csv("bazaar_books/realm_metadata.csv"),
}
active_table.set_tables(dfs)
print("Tables loaded:", list(dfs))

Tables loaded: ['caravan_accounts', 'guild_ledger', 'realm_metadata']


### Load + chunk the two demo documents

`pymupdf` (pdf) / `python-docx` (docx) — no LangChain loader wrapper.
Splitting: `RecursiveCharacterTextSplitter`, chunk_size=500/overlap=50
(reasonable defaults, not tuned — doesn't matter yet without embeddings).

In [13]:
def load_pdf(path: str) -> str:
    """Extract all text from a PDF file.

    Args:
        path: path to a .pdf file.

    Returns:
        The concatenated text of every page.
    """
    with pymupdf.open(path) as pdf:
        return "\n".join(page.get_text() for page in pdf)


def load_docx(path: str) -> str:
    """Extract all paragraph text from a .docx file.

    Args:
        path: path to a .docx file.

    Returns:
        The concatenated text of every paragraph.
    """
    doc = DocxDocument(path)
    return "\n".join(p.text for p in doc.paragraphs)


DOCUMENT_FILES = {
    "zau_al_makan_decree.docx": load_docx("bazaar_books/zau_al_makan_decree.docx"),
    "hammam_keeper_proclamation.pdf": load_pdf("bazaar_books/hammam_keeper_proclamation.pdf"),
}

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

document_chunks = [
    {"text": chunk, "source": source}
    for source, full_text in DOCUMENT_FILES.items()
    for chunk in splitter.split_text(full_text)
]

for c in document_chunks:
    print(f"[{c['source']}] {c['text'][:80]}...")

[zau_al_makan_decree.docx] Decree of Zau al-Makan Concerning the Silk Road Tariff
Year 717
In the year 717,...
[zau_al_makan_decree.docx] Let it be known that the caravan tariff upon all iron and steel goods carried by...
[hammam_keeper_proclamation.pdf] Proclamation of King Omar bin al-Nu'uman
Concerning the Bath-Attendant's Reward,...
[hammam_keeper_proclamation.pdf] of his service to the royal house.
This proclamation is sealed by order of King ...


### Naive search + `search_documents` tool

Word-overlap scoring, no vectors — proves the tool→agent→answer shape
before adding real embeddings in Stage 2. Same shape as
`execute_python_code`/`create_chart` (try/except, output truncation).

In [14]:
def naive_keyword_search(query: str, chunks: list, top_n: int = 3) -> list:
    """Rank chunks by word overlap with the query — no embeddings, no vectors.

    Args:
        query: the search query.
        chunks: list of {"text": ..., "source": ...} dicts.
        top_n: how many top-scoring chunks to return.

    Returns:
        The top_n chunks (same dict shape), highest overlap first, excluding
        chunks with zero overlap.
    """
    query_words = set(query.lower().split())
    scored = [
        (len(query_words & set(chunk["text"].lower().split())), chunk)
        for chunk in chunks
    ]
    scored = [(score, chunk) for score, chunk in scored if score > 0]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [chunk for _, chunk in scored[:top_n]]


@tool
def search_documents(query: str) -> str:
    """Search the loaded non-tabular documents (PDF/DOCX) for relevant text.

    Use this for questions about document content — decrees, proclamations,
    agreements — as opposed to numeric/tabular questions, which should use
    execute_python_code instead. Returns the most relevant chunks of text
    found, each tagged with its source file.

    Args:
        query: the search query — what you're looking for in the documents.

    Returns:
        The most relevant chunks of text, each prefixed with its source
        file name, or a message saying nothing relevant was found.
    """
    try:
        hits = naive_keyword_search(query, document_chunks)
    except Exception as e:
        return f"Error: {e}"

    if not hits:
        return "No relevant text found in the loaded documents for this query."

    output = "\n---\n".join(f"[{hit['source']}] {hit['text']}" for hit in hits)
    if len(output) > MAX_TOOL_OUTPUT_CHARS:
        return output[:MAX_TOOL_OUTPUT_CHARS] + "\n...\n[Output truncated — narrow your query.]"
    return output


# quick manual check — no LLM involved, just calling the tool function directly
print(search_documents.invoke({"query": "tariff Djinn-Forged Ironworks"}))

[zau_al_makan_decree.docx] Decree of Zau al-Makan Concerning the Silk Road Tariff
Year 717
In the year 717, in the season following the lifting of the siege, I, Zau al-Makan, son of Omar bin al-Nu'uman, do proclaim this decree to all caravan-masters and tax-collectors of the realm.
Whereas the guild known as Djinn-Forged Ironworks did furnish arms and siege-engines to the defenders of the Oasis of Whispering Sands without delay or excess price, and whereas their smiths labored through three winters without rest,
---
[zau_al_makan_decree.docx] Let it be known that the caravan tariff upon all iron and steel goods carried by Djinn-Forged Ironworks through the Peak of the Sleeping Djinn shall be reduced from the customary twelve parts in one hundred to but four parts in one hundred, for a term of five years from this decree.
This exemption applies to no other guild, and may be revoked should the Ironworks fail to supply the royal armories in time of need.
Sealed in the hall of the Oasis of

### Extend the existing agent

`search_documents` added alongside the other two tools on the same agent —
no new agent, no router. `build_system_prompt(dfs)` reused as-is, plus a
short documents section.

In [15]:
DOCUMENTS_SECTION = """\

## Documents available

You also have access to non-tabular documents (not part of `dfs`), listed
below by file name. For questions about their content — decrees,
proclamations, agreements — call `search_documents` instead of
`execute_python_code`. Never guess document content; always search for it.

{document_names}
"""

document_names = "\n".join(f"- {name}" for name in DOCUMENT_FILES)
system_prompt = build_system_prompt(dfs) + DOCUMENTS_SECTION.format(document_names=document_names)

agent = create_agent(
    model=model,
    tools=[execute_python_code, create_chart, search_documents],
    system_prompt=system_prompt,
)
print("Agent built with 3 tools: execute_python_code, create_chart, search_documents")

Agent built with 3 tools: execute_python_code, create_chart, search_documents


### Live test

Two document questions (each answerable only from its file) plus one
tabular question, to check for regressions on the existing tools.

In [16]:
for question in [
    "What tariff rate does Djinn-Forged Ironworks pay under Zau al-Makan's decree, and for how long?",
    "What reward did the hammam keeper receive from King Omar bin al-Nu'uman?",
    "Which realm had the highest net income?",
]:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Q: {question}")
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:
                print(f"  tool: {call['name']}  args: {call['args']}")
    print(f"A: {result['messages'][-1].content.strip()}")
    print("-" * 80)

Q: What tariff rate does Djinn-Forged Ironworks pay under Zau al-Makan's decree, and for how long?
  tool: search_documents  args: {'query': 'Zau al-Makan decree Djinn-Forged Ironworks tariff rate duration'}
A: The tariff rate for Djinn-Forged Ironworks under Zau al-Makan's decree is **4%** (reduced from the standard 12%), and the exemption lasts for **five years** from the decree date in Year 717.
--------------------------------------------------------------------------------
Q: What reward did the hammam keeper receive from King Omar bin al-Nu'uman?
  tool: search_documents  args: {'query': "reward hammam keeper King Omar bin al-Nu'uman"}
A: The hammam keeper received a reward of **120 dinars** from the royal treasury and was granted a stall in the market of the Forty Thieves Foundry, free of all guild dues for **ten years**, as stated in the proclamation by King Omar bin al-Nu'uman.
--------------------------------------------------------------------------------
Q: Which realm had 

### Known gap: no-answer / multi-hop queries not tested

Naive keyword search has no relevance threshold — a question with no real
answer in the documents, or one needing to combine several chunks, will
likely still return some weak match, and the model may answer confidently
anyway. Handling this well is above-average RAG territory, out of scope
for this micro-prototype. Stage 1 goal (tool + naive search + no regression
on the tabular tools) is met as-is.

### Quick test: mlx-omni-server embeddings

Probe call to confirm the embedding server works before building anything
on top of it. Model: `mlx-community/Qwen3-Embedding-0.6B-mxfp8`, served by
`mlx-omni-server` on the Mac (dimensionality: 1024).

Note: `OpenAIEmbeddings` tokenizes client-side by default (sends token ids,
not a string) — mlx-omni-server expects a raw string, hence
`check_embedding_ctx_length=False` below.

In [7]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="mlx-community/Qwen3-Embedding-0.6B-mxfp8",
    base_url="http://10.195.19.15:8090/v1",
    api_key="dummy",
    check_embedding_ctx_length=False,   # send raw strings, skip client-side tokenization
)

In [8]:
vector = embeddings.embed_query("test")
print(len(vector))  # prints the vector dimensionality

1024


## Step 14: Chroma — real vector search (Stage 2)

Replaces Step 13's word-overlap search with real embeddings + Chroma. Same
`document_chunks` shape as Step 13, built independently here (self-
contained). `Chroma.from_texts` embeds each chunk via the client above;
`similarity_search` replaces the naive ranking — nothing else in the
pipeline changes.

In [9]:
import pymupdf
from docx import Document as DocxDocument
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_pdf(path: str) -> str:
    with pymupdf.open(path) as pdf:
        return "\n".join(page.get_text() for page in pdf)


def load_docx(path: str) -> str:
    doc = DocxDocument(path)
    return "\n".join(p.text for p in doc.paragraphs)


DOCUMENT_FILES = {
    "zau_al_makan_decree.docx": load_docx("bazaar_books/zau_al_makan_decree.docx"),
    "hammam_keeper_proclamation.pdf": load_pdf("bazaar_books/hammam_keeper_proclamation.pdf"),
}

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts, metadatas = [], []
for source, full_text in DOCUMENT_FILES.items():
    for chunk in splitter.split_text(full_text):
        texts.append(chunk)
        metadatas.append({"source": source})

vectorstore = Chroma.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)
print(f"{len(texts)} chunks indexed")

4 chunks indexed


### Live test: same questions as Step 13, real vector search this time

In [10]:
for query in [
    "What tariff rate does Djinn-Forged Ironworks pay under Zau al-Makan's decree?",
    "What reward did the hammam keeper receive from King Omar bin al-Nu'man?",
]:
    hits = vectorstore.similarity_search(query, k=2)
    print(f"Q: {query}")
    for h in hits:
        print(f"  [{h.metadata['source']}] {h.page_content[:100]}...")
    print()

Q: What tariff rate does Djinn-Forged Ironworks pay under Zau al-Makan's decree?
  [zau_al_makan_decree.docx] Let it be known that the caravan tariff upon all iron and steel goods carried by Djinn-Forged Ironwo...
  [zau_al_makan_decree.docx] Decree of Zau al-Makan Concerning the Silk Road Tariff
Year 717
In the year 717, in the season follo...

Q: What reward did the hammam keeper receive from King Omar bin al-Nu'man?
  [hammam_keeper_proclamation.pdf] Proclamation of King Omar bin al-Nu'uman
Concerning the Bath-Attendant's Reward, Year 717
Let it be ...
  [zau_al_makan_decree.docx] Decree of Zau al-Makan Concerning the Silk Road Tariff
Year 717
In the year 717, in the season follo...



## Step 15: persistent Chroma collection (Case 2 — "we have a knowledge base")

Self-contained — Steps 13/14 don't need to run first.

Case 1 (Step 14) is ephemeral, rebuilt every run. Case 2 needs to survive
restarts — same `Chroma` class, adds `persist_directory`, writing to
`chroma_db/` (gitignored).

Four synthetic documents loaded (two new ones added to the original pair).

**Gotcha:** `Chroma.from_texts` on an existing `persist_directory`/
`collection_name` appends rather than replaces — re-running the build
duplicates chunks and degrades retrieval. Fixed by wiping the directory
before every rebuild. Verified across two separate processes that
persistence survives with no re-embedding needed.

### Utility: wipe the persisted collection (no rebuild)

Wipes `chroma_db/` to a genuinely empty state — e.g. before testing with a
different file set, so old uploads don't linger. The build cell below
already wipes and rebuilds from the 4 synthetic docs on every run; this
cell is for a blank state instead.

In [23]:
import shutil
from pathlib import Path

from chromadb.api.client import SharedSystemClient

PERSIST_DIR = "chroma_db"
shutil.rmtree(PERSIST_DIR, ignore_errors=True)
Path(PERSIST_DIR).mkdir(exist_ok=True)

# chromadb caches clients per-process (SharedSystemClient) -- without
# clearing it here, a client created against this path later in the same
# kernel session can reuse a stale connection to the now-deleted database
# and fail with "attempt to write a readonly database" (verified: this
# reproduced the exact error, and clearing the cache fixed it).
SharedSystemClient.clear_system_cache()
print(f"{PERSIST_DIR}/ wiped clean.")

chroma_db/ wiped clean.


In [11]:
import shutil
from pathlib import Path

import pandas as pd
import pymupdf
from docx import Document as DocxDocument
from langchain.agents import create_agent
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.api.client import SharedSystemClient

from finanalyticsagent import active_table
from finanalyticsagent.prompts import build_system_prompt
from finanalyticsagent.tools import MAX_TOOL_OUTPUT_CHARS, create_chart, execute_python_code

model = ChatOpenAI(
    model="Qwen/Qwen3-8B-MLX-4bit",
    base_url="http://10.195.19.15:8000/v1",
    api_key="dummy",
    temperature=0,
    max_tokens=8192,
)

dfs = {
    "caravan_accounts": pd.read_csv("bazaar_books/caravan_accounts.csv"),
    "guild_ledger": pd.read_csv("bazaar_books/guild_ledger.csv"),
    "realm_metadata": pd.read_csv("bazaar_books/realm_metadata.csv"),
}
active_table.set_tables(dfs)

embeddings = OpenAIEmbeddings(
    model="mlx-community/Qwen3-Embedding-0.6B-mxfp8",
    base_url="http://10.195.19.15:8090/v1",
    api_key="dummy",
    check_embedding_ctx_length=False,
)


def load_pdf(path: str) -> str:
    with pymupdf.open(path) as pdf:
        return "\n".join(page.get_text() for page in pdf)


def load_docx(path: str) -> str:
    doc = DocxDocument(path)
    return "\n".join(p.text for p in doc.paragraphs)


DOCUMENT_FILES = {
    "zau_al_makan_decree.docx": load_docx("bazaar_books/zau_al_makan_decree.docx"),
    "hammam_keeper_proclamation.pdf": load_pdf("bazaar_books/hammam_keeper_proclamation.pdf"),
    "taj_al_muluk_bazaar.docx": load_docx("bazaar_books/taj_al_muluk_bazaar.docx"),
    "aziz_reckoning.pdf": load_pdf("bazaar_books/aziz_reckoning.pdf"),
}

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts, metadatas = [], []
for source, full_text in DOCUMENT_FILES.items():
    for chunk in splitter.split_text(full_text):
        texts.append(chunk)
        metadatas.append({"source": source})

# Wipe before rebuilding — from_texts APPENDS to an existing persist_directory/
# collection_name rather than replacing it (see markdown above).
PERSIST_DIR = "chroma_db"
shutil.rmtree(PERSIST_DIR, ignore_errors=True)
Path(PERSIST_DIR).mkdir(exist_ok=True)
SharedSystemClient.clear_system_cache()  # see Step 15 utility cell above for why

vectorstore = Chroma.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadatas,
    collection_name="bazaar_books_kb",
    persist_directory=PERSIST_DIR,
)
print(f"Persisted {len(texts)} chunks to {PERSIST_DIR}/ (collection count: {vectorstore._collection.count()})")

Persisted 15 chunks to chroma_db/ (collection count: 15)


### Reload via a fresh client, no re-embedding

In [12]:
reloaded_store = Chroma(
    collection_name="bazaar_books_kb",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)
print(f"Reloaded {reloaded_store._collection.count()} chunks from disk")

hits = reloaded_store.similarity_search("What tariff does Djinn-Forged Ironworks pay?", k=1)
print(f"[{hits[0].metadata['source']}] {hits[0].page_content[:100]}...")

Reloaded 15 chunks from disk
[zau_al_makan_decree.docx] Let it be known that the caravan tariff upon all iron and steel goods carried by Djinn-Forged Ironwo...


### Wire the persisted store into the agent, live test

The part Step 15 was missing until now: everything above only exercised
`vectorstore`/`similarity_search` directly. This proves the persisted
collection actually answers correctly *through the chat model*, including
the exact gap found above (dibaj price) now fixed.

In [13]:
@tool
def search_documents(query: str) -> str:
    """Search the loaded non-tabular documents (PDF/DOCX) for relevant text.

    Use this for questions about document content — decrees, proclamations,
    tales, agreements — as opposed to numeric/tabular questions, which
    should use execute_python_code instead.

    Args:
        query: the search query.

    Returns:
        The most relevant chunks, each tagged with its source file, or an
        "Error: ..." message if the embedding/vector search backend fails.
    """
    try:
        hits = vectorstore.similarity_search(query, k=5)
    except Exception as e:
        return f"Error: {e}"
    if not hits:
        return "No relevant text found in the loaded documents for this query."
    output = "\n---\n".join(f"[{h.metadata['source']}] {h.page_content}" for h in hits)
    if len(output) > MAX_TOOL_OUTPUT_CHARS:
        return output[:MAX_TOOL_OUTPUT_CHARS] + "\n...\n[truncated]"
    return output


DOCUMENTS_SECTION = """\

## Documents available

You also have access to non-tabular documents (not part of `dfs`), listed
below by file name. For questions about their content — decrees,
proclamations, tales, agreements — call `search_documents` instead of
`execute_python_code`. Never guess document content; always search for it.

{document_names}
"""
document_names = "\n".join(f"- {name}" for name in DOCUMENT_FILES)
system_prompt = build_system_prompt(dfs) + DOCUMENTS_SECTION.format(document_names=document_names)

agent = create_agent(
    model=model,
    tools=[execute_python_code, create_chart, search_documents],
    system_prompt=system_prompt,
)

for question in [
    "In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?",
    "According to Aziz's reckoning, what was the total value of the day's trade?",
    "Which realm had the highest net income?",
]:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Q: {question}")
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:
                print(f"  tool: {call['name']}  args: {call['args']}")
    print(f"A: {result['messages'][-1].content.strip()}")
    print("-" * 80)

Q: In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?
  tool: search_documents  args: {'query': "price of the finest bolt of dibaj in taj al-muluk's story"}
A: In *Taj al-Muluk's* story, the finest bolt of dibaj was priced at **250 dinars**. This is explicitly stated in the description of the goods laid out on their table: "dibaj of deep crimson at two hundred and fifty dinars the bolt."
--------------------------------------------------------------------------------
Q: According to Aziz's reckoning, what was the total value of the day's trade?
  tool: search_documents  args: {'query': "total value of the day's trade"}
A: The total value of the day's trade, as reckoned by Aziz, was **1160 dinars**.
--------------------------------------------------------------------------------
Q: Which realm had the highest net income?
  tool: execute_python_code  args: {'code': "df = dfs['caravan_accounts]\nrealm_net_income = df.groupby('Realm')['Net_Income'].sum()\nprint(realm_

## Step 16: real user-driven document upload (PDF/DOCX)

Same pattern as Step 8's `ipywidgets.FileUpload` for CSV/XLSX — a button
opens the native file picker for a real `.pdf`/`.docx` (real bytes, not a
typed path). Adds the document to Step 15's persisted Chroma collection
and rebuilds the agent. Two cells, same reason as Step 8 (upload is
asynchronous).

In [30]:
import ipywidgets as widgets
from IPython.display import display

doc_upload_widget = widgets.FileUpload(accept=".pdf,.docx", multiple=False)
display(doc_upload_widget)

# Click the button above, pick a .pdf or .docx from your own filesystem,
# then run the next cell.

FileUpload(value=(), accept='.pdf,.docx', description='Upload')

### Process the upload, add it to the persisted collection, rebuild the agent

In [31]:
import io

if not doc_upload_widget.value:
    raise ValueError("No file selected yet -- click the button above and choose a file first.")

uploaded = doc_upload_widget.value[0]
filename = uploaded["name"]
file_bytes = uploaded["content"]

if filename.endswith(".pdf"):
    with pymupdf.open(stream=file_bytes, filetype="pdf") as pdf:
        new_text = "\n".join(page.get_text() for page in pdf)
elif filename.endswith(".docx"):
    new_text = "\n".join(p.text for p in DocxDocument(io.BytesIO(file_bytes)).paragraphs)
else:
    raise ValueError(f"Unsupported file type: {filename}")

new_chunks = splitter.split_text(new_text)
vectorstore.add_texts(texts=new_chunks, metadatas=[{"source": filename}] * len(new_chunks))
DOCUMENT_FILES[filename] = new_text
print(f"Added {len(new_chunks)} chunks from {filename!r} to the persisted collection "
      f"(total now {vectorstore._collection.count()})")

document_names = "\n".join(f"- {name}" for name in DOCUMENT_FILES)
system_prompt = build_system_prompt(dfs) + DOCUMENTS_SECTION.format(document_names=document_names)
agent = create_agent(
    model=model,
    tools=[execute_python_code, create_chart, search_documents],
    system_prompt=system_prompt,
)

print(
    "\n--- Data hygiene reminder ---\n"
    "The file you just uploaded is now embedded and persisted to chroma_db/.\n"
    "chroma_db/ is git-ignored, so nothing gets committed automatically -- but\n"
    "if that ever changes, make sure only synthetic data (bazaar_books/*) gets\n"
    "pushed, not anything real."
)

Added 72 chunks from 'Accor_URD_2023_EN_short_v_for_tests.pdf' to the persisted collection (total now 91)

--- Data hygiene reminder ---
The file you just uploaded is now embedded and persisted to chroma_db/.
chroma_db/ is git-ignored, so nothing gets committed automatically -- but
if that ever changes, make sure only synthetic data (bazaar_books/*) gets
pushed, not anything real.


### Ask about the newly uploaded document

In [32]:
question = "What does the newly uploaded document say? Summarize its key point."
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(f"Q: {question}")
print(f"A: {result['messages'][-1].content.strip()}")

Q: What does the newly uploaded document say? Summarize its key point.
A: The newly uploaded document is Accor's 2023 Universal Registration Document, highlighting their **Integrated Report**. Key points include:  
- Expanding **economy hotels** with affordability, conviviality, and simplicity (e.g., *ibis Styles Rotterdam Ahoy*, *Mercure Danang French Village*).  
- Targeting diverse travelers: "movers and shakers," young adventurers, and those seeking everyday adventure.  
- Scale: **641 hotels**, **65,060 rooms**, and **3,920 rooms under development** across 25 countries.  
- Strengthened **compliance efforts**: A 111% increase in whistleblower alerts in 2023, with a dedicated committee addressing concerns.  

The document balances growth ambitions with transparency and ethical governance.


## Step 17: RAG evaluation metrics (RAGAS)

`ragas` 0.4.3 evaluated against the Step 15 pipeline, same questions/answers
used as live tests. Self-contained.

**Why the workarounds below exist:**
- A `sys.modules` stub for `langchain_community.chat_models.vertexai` is
  required before `import ragas` — this ragas version still imports
  `ChatVertexAI` from a path that `langchain-community` no longer has
  (that class moved to `langchain-google-vertexai`). We never use
  VertexAI — the stub only satisfies the import.
- Metrics are called via `llm_factory` + per-sample async `.ascore()`, not
  the older batch `evaluate()` — passing current `ragas.metrics.collections`
  metrics into `evaluate()` raises `TypeError` in 0.4.3 (upstream bug,
  ragas#2624). `.ascore()` is the only path that currently works.
- Only `NonLLMStringSimilarity` (Levenshtein-based, no judge model) is
  asserted on. LLM-judge metrics (`Faithfulness`, `FactualCorrectness`) hit
  the same hidden-`<think>`-eats-`max_tokens` issue as our own agent on this
  local model — too unreliable to assert on.

Real pytest version: `tests/test_rag_metrics.py`.

In [1]:
import sys
import types

# ragas 0.4.3 still imports ChatVertexAI from a langchain_community path;
# that class moved to langchain-google-vertexai. Stub it -- we never use VertexAI.
_stub = types.ModuleType("langchain_community.chat_models.vertexai")
_stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = _stub

import pandas as pd
import pymupdf
from docx import Document as DocxDocument
from langchain.agents import create_agent
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas.metrics.collections import NonLLMStringSimilarity

from finanalyticsagent import active_table
from finanalyticsagent.prompts import build_system_prompt
from finanalyticsagent.tools import create_chart, execute_python_code

model = ChatOpenAI(
    model="Qwen/Qwen3-8B-MLX-4bit",
    base_url="http://10.195.19.15:8000/v1",
    api_key="dummy",
    temperature=0,
    max_tokens=8192,
)

dfs = {
    "caravan_accounts": pd.read_csv("bazaar_books/caravan_accounts.csv"),
    "guild_ledger": pd.read_csv("bazaar_books/guild_ledger.csv"),
    "realm_metadata": pd.read_csv("bazaar_books/realm_metadata.csv"),
}
active_table.set_tables(dfs)

embeddings = OpenAIEmbeddings(
    model="mlx-community/Qwen3-Embedding-0.6B-mxfp8",
    base_url="http://10.195.19.15:8090/v1",
    api_key="dummy",
    check_embedding_ctx_length=False,
)


def load_pdf(path: str) -> str:
    with pymupdf.open(path) as pdf:
        return "\n".join(page.get_text() for page in pdf)


def load_docx(path: str) -> str:
    doc = DocxDocument(path)
    return "\n".join(p.text for p in doc.paragraphs)


DOCUMENT_FILES = {
    "zau_al_makan_decree.docx": load_docx("bazaar_books/zau_al_makan_decree.docx"),
    "hammam_keeper_proclamation.pdf": load_pdf("bazaar_books/hammam_keeper_proclamation.pdf"),
    "taj_al_muluk_bazaar.docx": load_docx("bazaar_books/taj_al_muluk_bazaar.docx"),
    "aziz_reckoning.pdf": load_pdf("bazaar_books/aziz_reckoning.pdf"),
}

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts, metadatas = [], []
for source, full_text in DOCUMENT_FILES.items():
    for chunk in splitter.split_text(full_text):
        texts.append(chunk)
        metadatas.append({"source": source})

vectorstore = Chroma.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)


@tool
def search_documents(query: str) -> str:
    """Search the loaded non-tabular documents for relevant text."""
    hits = vectorstore.similarity_search(query, k=5)
    return "\n---\n".join(f"[{h.metadata['source']}] {h.page_content}" for h in hits)


document_names = "\n".join(f"- {name}" for name in DOCUMENT_FILES)
system_prompt = build_system_prompt(dfs) + f"\n\n## Documents available\n\n{document_names}\n"

agent = create_agent(
    model=model,
    tools=[execute_python_code, create_chart, search_documents],
    system_prompt=system_prompt,
)

EVAL_QUESTIONS = [
    {
        "question": "In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?",
        "reference": "The finest bolt of dibaj was priced at two hundred and fifty dinars.",
    },
    {
        "question": "According to Aziz's reckoning, what was the total value of the day's trade?",
        "reference": "The total value of the day's trade was one thousand one hundred and sixty dinars.",
    },
    {
        "question": "What reward did the hammam keeper receive from King Omar bin al-Nu'uman?",
        "reference": "The hammam keeper received 120 dinars and a stall in the Forty Thieves Foundry market, free of guild dues for ten years.",
    },
]
print(f"Agent + {len(texts)} chunks ready, {len(EVAL_QUESTIONS)} eval questions loaded.")

Agent + 15 chunks ready, 3 eval questions loaded.


In [2]:
metric = NonLLMStringSimilarity()


async def score_all():
    for item in EVAL_QUESTIONS:
        result = agent.invoke({"messages": [{"role": "user", "content": item["question"]}]})
        response = result["messages"][-1].content.strip()
        score = await metric.ascore(response=response, reference=item["reference"])
        print(f"Q: {item['question']}")
        print(f"A: {response}")
        print(f"  non_llm_string_similarity: {score.value:.3f}")
        print("-" * 80)


# Jupyter/IPython already runs its own event loop -- use top-level await
# (supported natively in notebook cells) instead of asyncio.run(), which
# raises "cannot be called from a running event loop" here.
await score_all()

Q: In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?
A: The price of the finest bolt of dibaj in Taj al-Muluk's story was **250 dinars**.
  non_llm_string_similarity: 0.432
--------------------------------------------------------------------------------
Q: According to Aziz's reckoning, what was the total value of the day's trade?
A: The total value of the day's trade, as reckoned by Aziz, was **1160 dinars**.
  non_llm_string_similarity: 0.605
--------------------------------------------------------------------------------
Q: What reward did the hammam keeper receive from King Omar bin al-Nu'uman?
A: The Hammam Keeper received a reward of **120 dinars** from the royal treasury and was granted a stall in the market of the Forty Thieves Foundry, free of all guild dues for ten years, as stated in the proclamation by King Omar bin al-Nu'uman.
  non_llm_string_similarity: 0.430
--------------------------------------------------------------------------------


**Revisit once upstream is fixed:** the two workarounds above are tied to
specific ragas 0.4.3 bugs (ragas#2745, ragas#2624), not permanent design
choices. Once ragas fixes the `ChatVertexAI` import and the
`evaluate()`/`collections` incompatibility, drop the `sys.modules` stub and
check whether `evaluate()` (or whatever the then-current recommended API
is) is simpler than the per-sample `.ascore()` loop used here — verify
against ragas's current docs first, don't assume this note stays accurate.

## Step 18: RAG via the extracted modules (Stage 2, done)

Same role as Step 10 for tables: proves `finanalyticsagent.documents`/
`tools`/`graph` work standalone, calling the real package instead of the
inline code from Steps 13-17 (which stay as-is, R&D history).

`build_agent(tables, document_files)` now optionally builds/reloads a
Chroma knowledge base and adds `search_documents` — chunk size, overlap,
`k`, and persistence paths live in one place (`documents.py`), no longer
duplicated between notebook and tests.

In [3]:
import pandas as pd

from finanalyticsagent import documents
from finanalyticsagent.graph import build_agent

dfs = {
    "caravan_accounts": pd.read_csv("bazaar_books/caravan_accounts.csv"),
    "guild_ledger": pd.read_csv("bazaar_books/guild_ledger.csv"),
    "realm_metadata": pd.read_csv("bazaar_books/realm_metadata.csv"),
}

document_files = {
    "zau_al_makan_decree.docx": documents.load_docx("bazaar_books/zau_al_makan_decree.docx"),
    "hammam_keeper_proclamation.pdf": documents.load_pdf("bazaar_books/hammam_keeper_proclamation.pdf"),
    "taj_al_muluk_bazaar.docx": documents.load_docx("bazaar_books/taj_al_muluk_bazaar.docx"),
    "aziz_reckoning.pdf": documents.load_pdf("bazaar_books/aziz_reckoning.pdf"),
}

agent = build_agent(dfs, document_files)

for question in [
    "In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?",
    "Which realm had the highest net income?",
]:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Q: {question}")
    print(f"A: {result['messages'][-1].content.strip()}")
    print("-" * 80)

Q: In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?
A: In *Taj al-Muluk's* story, the finest bolt of dibaj was priced at **250 dinars** per bolt, as described in the *taj_al_muluk_bazaar.docx* document.
--------------------------------------------------------------------------------
Q: Which realm had the highest net income?
A: The realm with the highest net income is **Garden of the Midnight Rose**.
--------------------------------------------------------------------------------


## Step 19: real file upload through the extracted modules

Verifies module wiring, not `app.py` — same role as Step 8/16's upload
widgets, but the loading/dispatch calls the real package
(`finanalyticsagent.documents`/`graph`) instead of duplicating parsing code
in the notebook. Self-contained; does not depend on Steps 13-18.

In [4]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from finanalyticsagent.graph import build_agent

dfs = {
    "caravan_accounts": pd.read_csv("bazaar_books/caravan_accounts.csv"),
    "guild_ledger": pd.read_csv("bazaar_books/guild_ledger.csv"),
    "realm_metadata": pd.read_csv("bazaar_books/realm_metadata.csv"),
}

upload_widget = widgets.FileUpload(accept=".csv,.xlsx,.pdf,.docx", multiple=True)
display(upload_widget)

# Click the button above, pick one or more .csv/.xlsx/.pdf/.docx files,
# then run the next cell.

FileUpload(value=(), accept='.csv,.xlsx,.pdf,.docx', description='Upload', multiple=True)

### Dispatch by extension (plain Python, not LLM) and build the agent through the real modules

In [5]:
import io
import tempfile
from pathlib import Path

from finanalyticsagent import documents

if not upload_widget.value:
    raise ValueError("No file selected yet -- click the button above and choose a file first.")

tables = dict(dfs)
document_files = {}

for uploaded in upload_widget.value:
    name = uploaded["name"]
    file_bytes = uploaded["content"]

    if name.endswith(".csv"):
        tables[Path(name).stem] = pd.read_csv(io.BytesIO(file_bytes))
    elif name.endswith(".xlsx"):
        tables[Path(name).stem] = pd.read_excel(io.BytesIO(file_bytes))
    elif name.endswith(".pdf"):
        with tempfile.NamedTemporaryFile(suffix=".pdf") as tmp:
            tmp.write(file_bytes)
            tmp.flush()
            document_files[name] = documents.load_pdf(tmp.name)
    elif name.endswith(".docx"):
        with tempfile.NamedTemporaryFile(suffix=".docx") as tmp:
            tmp.write(file_bytes)
            tmp.flush()
            document_files[name] = documents.load_docx(tmp.name)
    else:
        raise ValueError(f"Unsupported file type: {name}")

agent = build_agent(tables, document_files or None)

print(f"Tables loaded: {list(tables)}")
print(f"Documents loaded: {list(document_files)}")

Tables loaded: ['caravan_accounts', 'guild_ledger', 'realm_metadata']
Documents loaded: ['Accor_URD_2023_EN_short_v_for_tests.pdf']


### Ask a question about the uploaded file

The question below should be edited to match the file uploaded above.

In [6]:
question = "Summarize what you just loaded — what's in the newly uploaded file(s)?"

result = agent.invoke({"messages": [{"role": "user", "content": question}]})

print(f"Q: {question}")
for message in result["messages"]:
    if message.type == "ai":
        for call in message.tool_calls:
            print(f"  tool: {call['name']}  args: {call['args']}")
print(f"A: {result['messages'][-1].content.strip()}")

Q: Summarize what you just loaded — what's in the newly uploaded file(s)?
A: The data includes three structured tables and one document. The tables cover financial metrics (Operating Income, EBITDA, Net Income) by realm and guild, guild performance metrics (market share, employee counts), and realm regional classifications. The document appears to be a PDF related to Accor's URD (Unified Reporting Document) for 2023, though its content isn't fully visible here.


## Step 20: mixed/ambiguous query behavior — baseline (before any tabular-priority prompt change)

Loads tables and documents together via hardcoded paths (not the upload
widget — this is about testing tool-selection behavior, not the upload
path already covered by Step 19). Self-contained, own agent build.

Four questions, in increasing order of ambiguity:
1. Purely tabular (control)
2. Purely document-based (control)
3. Mixed — genuinely needs both tools to fully answer
4. Ambiguous — plausible reading in both domains, no clear cue which tool applies

This is a baseline recorded *before* any prompt change biasing the agent
toward tabular tools over `search_documents` — the planned next step is to
compare this against the same four questions after that change.

In [7]:
import pandas as pd

from finanalyticsagent import documents
from finanalyticsagent.graph import build_agent

dfs = {
    "caravan_accounts": pd.read_csv("bazaar_books/caravan_accounts.csv"),
    "guild_ledger": pd.read_csv("bazaar_books/guild_ledger.csv"),
    "realm_metadata": pd.read_csv("bazaar_books/realm_metadata.csv"),
}

document_files = {
    "zau_al_makan_decree.docx": documents.load_docx("bazaar_books/zau_al_makan_decree.docx"),
    "hammam_keeper_proclamation.pdf": documents.load_pdf("bazaar_books/hammam_keeper_proclamation.pdf"),
    "taj_al_muluk_bazaar.docx": documents.load_docx("bazaar_books/taj_al_muluk_bazaar.docx"),
    "aziz_reckoning.pdf": documents.load_pdf("bazaar_books/aziz_reckoning.pdf"),
}

agent = build_agent(dfs, document_files)

QUESTIONS = {
    "tabular (control)": "Which realm had the highest net income?",
    "document (control)": "In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?",
    "mixed (needs both)": (
        "What was the total value of Aziz's day's trade according to his "
        "reckoning, and which realm's net income is closest to that amount?"
    ),
    "ambiguous (no clear cue)": "What was the total trade value?",
}

for label, question in QUESTIONS.items():
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"=== {label} ===")
    print(f"Q: {question}")
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:
                print(f"  tool: {call['name']}  args: {call['args']}")
    print(f"A: {result['messages'][-1].content.strip()}")
    print("-" * 80)

=== tabular (control) ===
Q: Which realm had the highest net income?
  tool: execute_python_code  args: {'code': "df = dfs['caravan_accounts]\nrealm_net_income = df.groupby('Realm')['Net_Income'].sum()\nprint(realm_net_income.idxmax())"}
  tool: execute_python_code  args: {'code': "df = dfs['caravan_accounts']\nrealm_net_income = df.groupby('Realm')['Net_Income'].sum()\nprint(realm_net_income.idxmax())"}
A: The realm with the highest net income is **Garden of the Midnight Rose**.
--------------------------------------------------------------------------------
=== document (control) ===
Q: In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?
  tool: search_documents  args: {'query': "price of the finest bolt of dibaj in Taj al-Muluk's story"}
A: In *Taj al-Muluk's* story, the finest bolt of dibaj was priced at **250 dinars** per bolt, as described in the *taj_al_muluk_bazaar.docx* document.
-----------------------------------------------------------------------------

## Step 21: regression test — same questions, Gemma 3 backend

Same questions and `NonLLMStringSimilarity` check as Steps 15-17, run
against `gemma-3-12b-it-4bit` instead of Qwen3 (same server, different
model loaded) — confirms the pipeline still works after the model swap.
No LLM-judge metric here; `Faithfulness` across Qwen3/Qwen3.5/Gemma3 is a
separate future step.

Non-agentic — retrieves context directly via `similarity_search`, then one
plain `model.invoke(...)` call, bypassing `create_agent`/tools. Each result
prints Qwen3's answer from Step 15/16 alongside Gemma3's for comparison.

In [ ]:
import sys
import types

# Must run before importing ragas — see Step 17 for why.
_stub = types.ModuleType("langchain_community.chat_models.vertexai")
_stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = _stub

from langchain_openai import ChatOpenAI
from ragas.metrics.collections import NonLLMStringSimilarity

from finanalyticsagent import documents

GEMMA_MODEL_NAME = "mlx-community/gemma-3-12b-it-4bit"
BASE_URL = "http://10.195.19.15:8000/v1"

gemma_model = ChatOpenAI(model=GEMMA_MODEL_NAME, base_url=BASE_URL, api_key="dummy", temperature=0)
vectorstore = documents.ensure_canonical_knowledge_base()

# question -> (reference used for scoring, real Qwen3 answer already recorded
# in this notebook -- Step 15/16 -- for a direct side-by-side comparison)
EVAL_QUESTIONS = [
    {
        "question": "In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?",
        "reference": "The finest bolt of dibaj was priced at two hundred and fifty dinars.",
        "qwen3_answer": (
            "In *Taj al-Muluk's* story, the finest bolt of dibaj was priced at "
            "**250 dinars**. This is explicitly stated in the description of the "
            "goods laid out on their table: \"dibaj of deep crimson at two hundred "
            "and fifty dinars the bolt.\""
        ),
    },
    {
        "question": "According to Aziz's reckoning, what was the total value of the day's trade?",
        "reference": "The total value of the day's trade was one thousand one hundred and sixty dinars.",
        "qwen3_answer": "The total value of the day's trade, as reckoned by Aziz, was **1160 dinars**.",
    },
    {
        "question": "What reward did the hammam keeper receive from King Omar bin al-Nu'uman?",
        "reference": "The hammam keeper received 120 dinars and a stall in the Forty Thieves Foundry market, free of guild dues for ten years.",
        "qwen3_answer": (
            "The hammam keeper received a reward of **120 dinars** from the royal "
            "treasury and was granted a stall in the market of the Forty Thieves "
            "Foundry, free of all guild dues for **ten years**, as stated in the "
            "proclamation by King Omar bin al-Nu'uman."
        ),
    },
]

similarity_metric = NonLLMStringSimilarity()


async def run_eval():
    for item in EVAL_QUESTIONS:
        hits = vectorstore.similarity_search(item["question"], k=documents.SEARCH_K)
        context_block = "\n---\n".join(h.page_content for h in hits)

        prompt = (
            "Answer the question using only the context below. If the "
            "context doesn't contain the answer, say so plainly.\n\n"
            f"Context:\n{context_block}\n\nQuestion: {item['question']}"
        )
        response = gemma_model.invoke(prompt).content.strip()

        sim_score = await similarity_metric.ascore(response=response, reference=item["reference"])

        print(f"Q: {item['question']}")
        print(f"Qwen3 answered (recorded earlier): {item['qwen3_answer']}")
        print(f"Gemma3 answers now: {response}")
        print(f"  non_llm_string_similarity (Gemma3 vs reference): {sim_score.value:.3f}")
        print("-" * 80)


# Jupyter already runs its own event loop -- top-level await, ыnot asyncio.run().
await run_eval()

Q: In Taj al-Muluk's story, what was the price of the finest bolt of dibaj?
Qwen3 answered (recorded earlier): In *Taj al-Muluk's* story, the finest bolt of dibaj was priced at **250 dinars**. This is explicitly stated in the description of the goods laid out on their table: "dibaj of deep crimson at two hundred and fifty dinars the bolt."
Gemma3 answers now: The finest bolt of dibaj was sold for two hundred and fifty dinars.
  non_llm_string_similarity (Gemma3 vs reference): 0.882
--------------------------------------------------------------------------------
Q: According to Aziz's reckoning, what was the total value of the day's trade?
Qwen3 answered (recorded earlier): The total value of the day's trade, as reckoned by Aziz, was **1160 dinars**.
Gemma3 answers now: According to Aziz's reckoning, the total value of the day's trade was one thousand one hundred and sixty dinars.
  non_llm_string_similarity (Gemma3 vs reference): 0.714
--------------------------------------------------